# Notebook 05 - Pipeline RAG

**MixCraft** - EFREI M1 DE&IA - Adam Beloucif & Emilien Morice - Janvier 2026

## Objectifs

1. Construire l'index FAISS pour le retrieval rapide
2. Implementer le pipeline RAG complet
3. Tester le guardrail semantique (seuil 0.40)
4. Analyser le cache MD5 et les gains de performance
5. Evaluer la qualite RAG vs generation sans contexte

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path

EFREI_NAVY = '#163767'
EFREI_PINK = '#FF43B8'
EFREI_BLUE = '#0C78B4'

from src.data_loader import load_all_datasets
from src.embeddings import EmbeddingEngine
from src.recommender import CocktailRecommender
from src.rag_pipeline import RAGPipeline

df = load_all_datasets()
engine = EmbeddingEngine(cache=True)
rec = CocktailRecommender(engine=engine)
rec.fit(df)
print(f'Systeme pret : {len(df)} cocktails indexes')

## 1. Construction du pipeline RAG

In [ ]:
rag = RAGPipeline(recommender=rec, guardrail_threshold=0.40)
print(f'Pipeline RAG initialise.')
print(f'Seuil guardrail : {rag.guardrail_threshold}')

## 2. Test du guardrail semantique

In [ ]:
# Requetes in-domain (doivent passer)
in_domain = [
    'cocktail avec vodka et citron',
    'quelque chose de frais et sucre',
    'rhum menthe mojito',
    'boisson pour ete fruité',
    'gin tonic cucumber',
]

# Requetes hors-domain (doivent etre rejetees)
out_of_domain = [
    'repare mon velo',
    'quelle est la capitale de la France',
    'recommande-moi un film de science fiction',
    'aide moi a coder en Python',
    'meteo de demain a Paris',
    'recette de gateau au chocolat',
]

print('=== Test Guardrail (seuil=0.40) ===\n')
print('--- REQUETES IN-DOMAIN (doivent PASSER) ---')
for q in in_domain:
    result = rag._check_guardrail(q)
    status = 'PASSE' if result['pass'] else 'REJETE'
    icon = 'OK' if result['pass'] else 'ERR'
    print(f'[{icon}] {status:7s} | sim={result["max_similarity"]:.3f} | "{q}"')

print('\n--- REQUETES HORS-DOMAIN (doivent etre REJETEES) ---')
for q in out_of_domain:
    result = rag._check_guardrail(q)
    status = 'PASSE' if result['pass'] else 'REJETE'
    icon = 'ERR' if result['pass'] else 'OK'
    print(f'[{icon}] {status:7s} | sim={result["max_similarity"]:.3f} | "{q}"')

In [ ]:
# Courbe similarite pour differents seuils
all_queries = [(q, True) for q in in_domain] + [(q, False) for q in out_of_domain]
sims = []
for q, _ in all_queries:
    r = rag._check_guardrail(q)
    sims.append(r['max_similarity'])

fig, ax = plt.subplots(figsize=(12, 5))
colors = [EFREI_BLUE if label else EFREI_PINK for _, label in all_queries]
ax.bar(range(len(all_queries)), sims, color=colors, edgecolor='white')
ax.axhline(0.40, color=EFREI_NAVY, linestyle='--', linewidth=2, label='Seuil guardrail (0.40)')
ax.set_xticks(range(len(all_queries)))
labels = [q[:20]+'...' if len(q) > 20 else q for q, _ in all_queries]
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Similarite cosinus max')
ax.set_title('Guardrail semantique : scores de similarite par requete', fontweight='bold', color=EFREI_NAVY)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=EFREI_BLUE, label='In-domain (attendu : passer)'),
    Patch(facecolor=EFREI_PINK, label='Hors-domain (attendu : rejeter)'),
    plt.Line2D([0], [0], color=EFREI_NAVY, linestyle='--', label='Seuil 0.40'),
]
ax.legend(handles=legend_elements, fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../assets/fig_guardrail.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Pipeline RAG complet

In [ ]:
# Test pipeline complet
test_rag_queries = [
    'un cocktail tropical pour une fete d\'ete',
    'quelque chose de frais avec citron vert et menthe',
    'cocktail fort et classique pour un amateur de whiskey',
]

for query in test_rag_queries:
    print(f'\nRequete : "{query}"')
    print('-' * 60)
    
    t0 = time.time()
    result = rag.query(query, top_k=3, generate=False)  # generate=False pour demo sans GPU
    t1 = time.time()
    
    print(f'Statut         : {result["status"]}')
    print(f'Sim. max       : {result["max_similarity"]:.3f}')
    print(f'Temps reponse  : {(t1-t0)*1000:.0f}ms')
    print(f'Depuis cache   : {result["cached"]}')
    
    if result['retrieved_cocktails']:
        print('Cocktails retrieved :')
        for c in result['retrieved_cocktails'][:3]:
            name = c['name'] if isinstance(c, dict) else c.name
            score = c['similarity_score'] if isinstance(c, dict) else c.similarity_score
            print(f'  - {name} (score: {score:.3f})')

## 4. Test du cache MD5

In [ ]:
# Tester le gain de latence grace au cache
test_query_cache = 'cocktail frais et fruité avec agrumes'

print('Test 1 (premier appel - calcul) :')
t0 = time.time()
r1 = rag.query(test_query_cache, generate=False)
t1 = time.time()
print(f'  Temps : {(t1-t0)*1000:.0f}ms | Cache: {r1["cached"]}')

print('\nTest 2 (second appel - meme requete) :')
t0 = time.time()
r2 = rag.query(test_query_cache, generate=False)
t1 = time.time()
print(f'  Temps : {(t1-t0)*1000:.0f}ms | Cache: {r2["cached"]}')

print('\nNote : le cache agit sur la generation de texte (pas le retrieval).')
print('La generation coutant ~500ms via API, le cache apporte un gain significatif.')